# Lab 4: Particle in a Box and Computational Chemistry

## Purpose

In this lab you will:

**Chemistry:**
- Understand how boundary conditions lead to quantized energy levels
- Connect the particle-in-a-box model to molecular orbital energies
- Learn how empirical models can approximate expensive quantum calculations

**Coding:**
- Practice writing functions with multiple parameters
- Use curve fitting to optimize model parameters
- Run quantum chemistry calculations with Psi4

**Real-world connection:** The particle-in-a-box model explains why conjugated molecules (like beta-carotene) absorb specific colors of light. The empirical modeling approach you'll learn is used throughout computational chemistry to develop fast, accurate approximations.

## Estimated Time: 75-90 minutes

## Success Criteria

By the end of this lab, you should be able to:
- [ ] Identify which k values satisfy PIB boundary conditions
- [ ] Write the PIB energy formula and explain its dependence on n and L
- [ ] Create molecules using SMILES notation
- [ ] Run an SCF calculation and interpret orbital energies
- [ ] Fit an empirical model to quantum mechanical results

---
## Libraries

Run this cell first to import all required packages.

In [ ]:
# Standard scientific computing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Interactive widgets
import ipywidgets as widgets

# Curve fitting
from scipy.optimize import curve_fit

# File handling
import os
import shutil

# Computational chemistry packages
import psi4
from rdkit import Chem
from rdkit.Chem import AllChem
import py3Dmol
import fortecubeview

# Helper functions for this lab
from pib_helper import (
    plot_wave,
    interactive_pib_plot,
    pib_energy_plot,
    xyz_from_smiles,
    show_molecule,
    create_psi4_molecule,
    fit_function,
    get_gaps,
    calculate_r_squared,
    pib_fit_plot
)

---
# Warmup: Curve Fitting

Fitting curves to data is a fundamental skill in scientific analysis. In this warmup, you'll practice fitting different functional forms to a mystery dataset.

**Your task:** Load the mystery curve data and find a function that fits it well.

### Worked Example

Here's how to define a function for curve fitting. The first argument must be `x`, and subsequent arguments are the parameters to be optimized.

In [ ]:
# Load the mystery curve data
mystery_curve = pd.read_csv('data/mystery_curve.csv')
plt.scatter(mystery_curve['x'], mystery_curve['y'])
plt.xlabel('x')
plt.ylabel('y')
plt.title('Mystery Curve')
plt.show()

In [ ]:
# Example: A linear function
def my_linear_function(x, m, b):
    """Linear function: y = mx + b"""
    y = m * x + b
    return y

# Try fitting it - [1, 1] are initial guesses for m and b
fit_function(my_linear_function, [1, 1], mystery_curve)

### Your Turn (10 points)

The linear function doesn't fit well. Define a better function and fit it to the data.

**Useful functions:** `np.abs()`, `np.exp()`, `np.sin()`, `np.cos()`, `**` (power operator)

**Hint:** Look at the shape of the data. Is it symmetric? Does it have a peak or valley?

In [ ]:
# Define your function here
def my_better_function(x, a, b, c):
    """Your function description here"""
    # YOUR CODE HERE
    y = None  # Replace with your formula
    return y

In [ ]:
# Test your function
fit_function(my_better_function, [1, 1, 1], mystery_curve)

---
# Part 1: Particle in a Box

The particle in a box is the simplest quantum mechanical system. Understanding it builds intuition for more complex systems.

## 1.1 Solving the Schrödinger Equation

For a particle confined in a box of length L with infinite potential walls:

$$V(x) = \begin{cases} 0, & 0 \leq x \leq L \\ \infty, & \text{otherwise} \end{cases}$$

The time-independent Schrödinger equation inside the box becomes:

$$-\frac{\hbar^2}{2m}\frac{d^2\psi(x)}{dx^2} = E\psi(x)$$

**Three constraints determine the allowed wavefunctions:**
1. **Boundary conditions:** $\psi(0) = \psi(L) = 0$ (wavefunction must be zero at the walls)
2. **Continuity:** The wavefunction must be continuous
3. **Eigenfunction:** $\psi(x)$ must be proportional to its second derivative

### Explore: Finding Allowed k Values (5 points)

The general solution has the form $\psi(x) = A\sin(kx)$. Use the interactive plot below to find which values of k satisfy the boundary conditions.

**Tip:** Use the arrow keys for fine control of the slider.

In [ ]:
# Explore which k values satisfy the boundary conditions
interactive_pib_plot()

### Short Response Questions (15 points)

**Q1.1 (5 pts):** What values of k gave a wavefunction that satisfies the boundary conditions? List at least 3 values.

**Q1.2 (5 pts):** Write a formula relating k to the box length L and the quantum number n (where n = 1, 2, 3, ...).

**Q1.3 (5 pts, Bonus):** The plot shows two complex exponential functions ($e^{ikx}$ and $e^{-ikx}$) and their sum. What is the physical significance of these traveling wave solutions?

*Your answers here:*

Q1.1:

Q1.2:

Q1.3:

---
## 1.2 Energy Levels of the Particle in a Box

Now that we know the allowed wavefunctions, we can find the corresponding energies.

Since $\frac{d^2}{dx^2}\sin(kx) = -k^2\sin(kx)$, substituting into the Schrödinger equation gives:

$$E_n = \frac{\hbar^2 k^2}{2m} = \frac{\hbar^2}{2m}\frac{\pi^2 n^2}{L^2} = \frac{h^2 n^2}{8mL^2}$$

where $n = 1, 2, 3, ...$ is the quantum number.

### Target Practice: Write the Energy Function (10 points)

Complete the `pib_energy()` function below. We'll use **atomic units** where:
- $\hbar = 1$
- $m_e = 1$ (electron mass)
- $h = 2\pi$ (Planck's constant)
- 1 Angstrom = 1.8889 atomic units

In [ ]:
def pib_energy(n, L_angs=1.5):
    """
    Calculate energy levels for a particle in a 1D box using atomic units.
    
    Parameters:
        n (int): Quantum number (n = 1, 2, 3, ...)
        L_angs (float): Length of the box in Angstroms
        
    Returns:
        energy (float): Energy in atomic units (Hartree)
    """
    # Convert length from Angstroms to atomic units
    L_au = L_angs * 1.8889
    
    # Constants in atomic units
    h_au = 2 * np.pi      # Planck's constant
    m_electron_au = 1     # Electron mass
    
    # YOUR CODE HERE: Calculate energy using E = h^2 * n^2 / (8 * m * L^2)
    energy = None  # Replace with your formula
    
    return energy

In [ ]:
# Test your function - should give approximately 0.66 for n=1, L=1.5 Angstroms
print(f"E(n=1, L=1.5 Å) = {pib_energy(1, 1.5):.4f} Hartree")
print(f"E(n=2, L=1.5 Å) = {pib_energy(2, 1.5):.4f} Hartree")

In [ ]:
# Visualize how energy depends on n and L
pib_energy_plot(pib_energy)

### Short Response Questions (15 points)

**Q1.4 (5 pts):** How does visualizing the energy levels in 3D help you understand the $E \propto n^2/L^2$ relationship?

**Q1.5 (5 pts):** Predict: If $n = L$ (numerically), how would the energy change as n increases? What about if $L = n - 1$?

**Q1.6 (5 pts):** The quantum number n equals the number of antinodes in the wavefunction. How does energy relate to the number of nodes?

*Your answers here:*

Q1.4:

Q1.5:

Q1.6:

---
# Part 2: Computational Chemistry

Now we'll move from the idealized particle-in-a-box to real molecular calculations. We'll investigate how the orbital energies of conjugated alkenes compare to the PIB model.

## 2.1 Specifying Molecules with SMILES

**SMILES (Simplified Molecular Input Line Entry System)** lets us specify molecules as text strings:

| Rule | Example | Molecule |
|------|---------|----------|
| Implicit hydrogens | `C` | Methane |
| Chains | `CCO` | Ethanol |
| Double bonds | `C=C` | Ethene |
| Triple bonds | `C#C` | Ethyne |
| Branches | `CC(C)O` | Isopropanol |
| Rings | `C1CCCCC1` | Cyclohexane |

### Worked Example

In [ ]:
# Example: Display ethanol
show_molecule("CCO")

### Your Turn (10 points)

Create and display the following molecules using SMILES notation:
1. Benzene (hint: aromatic ring)
2. Hexane (6-carbon chain)
3. 2-Butanol (branched alcohol)
4. Butadiene (conjugated double bonds)

In [ ]:
# YOUR CODE HERE: Display each molecule
# Example: show_molecule("your_smiles_here")


### Short Response Questions (10 points)

**Q2.1 (5 pts):** Why is representing molecules as SMILES strings useful for computational chemistry? When might this be preferable to a graphical editor?

**Q2.2 (5 pts):** Give an example of structural information that is NOT explicitly defined in a basic SMILES string.

*Your answers here:*

Q2.1:

Q2.2:

---
## 2.2 Computing Orbital Energies with Hartree-Fock

The **Hartree-Fock (HF)** method, also called **Self-Consistent Field (SCF)**, is a foundational quantum chemistry method.

**High-level algorithm:**
1. Start with an initial guess for molecular orbitals
2. For each orbital, minimize the energy considering:
   - Kinetic energy
   - Electron-nucleus attraction  
   - Electron-electron repulsion
   - Exchange (Pauli exclusion)
3. Repeat until orbitals don't change (convergence)

We'll use **Psi4** to calculate orbital energies for octatetraene and compare to the PIB model.

### Worked Example: Psi4 Calculation

Study this example to understand the workflow.

In [ ]:
# ============================================
# SUBGOAL: Define the molecule
# ============================================
ethene_smiles = "C=C"
ethene_mol = create_psi4_molecule(ethene_smiles)

# ============================================
# SUBGOAL: Configure the calculation
# ============================================
psi4.set_output_file('output.dat', False)  # Suppress verbose output
psi4.set_memory('1 GB')
theory = 'SCF/STO-3G'  # Method/basis set

# ============================================
# SUBGOAL: Run and extract results
# ============================================
energy, wfn = psi4.energy(theory, return_wfn=True, molecule=ethene_mol)
print(f"Total energy: {energy:.6f} Hartree")

# ============================================
# SUBGOAL: Visualize and interpret
# ============================================
orbital_energies = wfn.epsilon_a().np  # Get orbital energies as numpy array
print(f"Number of orbitals: {len(orbital_energies)}")
print(f"HOMO energy: {orbital_energies[7]:.4f} Hartree")  # 8 electrons = orbital 7 (0-indexed)

### Your Turn: Octatetraene (15 points)

Octatetraene (`C=CC=CC=CC=C`) is a conjugated molecule with pi electrons delocalized along the carbon chain - similar to a particle in a box!

Complete the code below to run an SCF calculation on octatetraene.

In [ ]:
# ============================================
# SUBGOAL: Define the molecule
# ============================================
octatetraene_smiles = None  # YOUR CODE: Write the SMILES string
psi4_mol = None  # YOUR CODE: Use create_psi4_molecule()

# ============================================
# SUBGOAL: Configure the calculation
# ============================================
psi4.set_output_file('output.dat', False)
psi4.set_memory('1 GB')
my_theory = 'SCF/STO-3G'

# ============================================
# SUBGOAL: Run and extract results
# ============================================
energy, wfn = psi4.energy(my_theory, return_wfn=True, molecule=psi4_mol)
print(f"Total energy: {energy:.6f} Hartree")

In [ ]:
# ============================================
# SUBGOAL: Visualize and interpret
# ============================================
plt.scatter(range(len(wfn.epsilon_a().np)), wfn.epsilon_a().np)
plt.title('Orbital Energies of Octatetraene')
plt.ylabel('Energy [Hartree]')
plt.xlabel('Orbital Index')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='E = 0')
plt.legend()
plt.show()

# Uncomment to zoom in on valence orbitals:
# plt.xlim(10, 50)
# plt.ylim(-2, 2)

### Short Response Questions (15 points)

**Q2.3 (5 pts):** How is the Hartree-Fock equation similar to the Schrödinger equation for PIB? How is it different?

**Q2.4 (5 pts):** Why might the orbital energies differ from the simple $E \propto n^2$ prediction of the PIB model?

**Q2.5 (5 pts):** Looking at your plot, is there a range of orbital indices where the energies approximately follow a quadratic pattern?

*Your answers here:*

Q2.3:

Q2.4:

Q2.5:

---
## 2.3 Visualizing Computed Orbitals

We can visualize the molecular orbitals to identify which are $\pi$ orbitals (the ones most similar to PIB wavefunctions).

**$\pi$ orbitals have:** Electron density above and below the molecular plane, with a node in the plane.

**$\sigma$ orbitals have:** Electron density along the bond axis.

In [ ]:
# Create directory for orbital data
os.makedirs('cubes', exist_ok=True)

In [ ]:
# Generate cube files for orbitals 25-34 (near the HOMO-LUMO region)
psi4.set_options({
    'CUBEPROP_TASKS': ['orbitals'],
    'CUBEPROP_FILEPATH': 'cubes',
    'CUBEPROP_ORBITALS': list(range(25, 35))
})
psi4.cubeprop(wfn)

In [ ]:
# Visualize the orbitals - use the dropdown to explore different orbitals
view = fortecubeview.plot('cubes', colorscheme='wow')

### Target Practice: Identify π Orbitals (15 points)

Examine the orbitals and identify which ones are $\pi$ orbitals. For each ground-state (occupied) $\pi$ orbital, include the corresponding excited-state (unoccupied) $\pi$ orbital.

**How to tell:** $\pi$ orbitals have lobes above and below the molecular plane. Count the nodes to determine the ordering.

In [ ]:
# YOUR CODE: List the indices of the pi orbitals you identified
# Example: pi_indices = [26, 27, 28, ...]
pi_indices = []  # Replace with your list

In [ ]:
# Plot the pi orbital energies
# Note: We subtract 1 because Python uses 0-based indexing
indices = np.array(pi_indices) - 1
x = range(1, len(pi_indices) + 1)  # Quantum number (1, 2, 3, ...)
y = wfn.epsilon_a().np[indices]    # Orbital energies

plt.scatter(x, y, s=80, label='Pi orbital energies')

# Fit a quadratic to compare with PIB (E ∝ n²)
a, b, c = np.polyfit(x, y, 2)
x_fit = np.linspace(1, len(pi_indices), 100)
y_fit = a * x_fit**2 + b * x_fit + c
plt.plot(x_fit, y_fit, 'r--', label=f'Quadratic fit: {a:.3f}n² + {b:.2f}n + {c:.2f}')

plt.legend()
plt.title('Pi Orbital Energies')
plt.ylabel('Energy [Hartree]')
plt.xlabel('Pi orbital number (n)')
plt.show()

In [ ]:
# Clean up the cube files
shutil.rmtree('cubes')

### Short Response Questions (15 points)

**Q2.6 (5 pts):** When else might visualizing molecular orbitals help you understand a chemical system?

**Q2.7 (5 pts):** How do the computed $\pi$ orbitals compare to the PIB wavefunctions? What similarities and differences do you observe?

**Q2.8 (5 pts):** How well does the quadratic fit describe the $\pi$ orbital energies? If there are discrepancies, what might cause them?

*Your answers here:*

Q2.6:

Q2.7:

Q2.8:

---
# Part 3: Making an Empirical Model

While SCF calculations are accurate, they're computationally expensive. **Empirical models** use simple equations with fitted parameters to approximate expensive calculations.

**Our goal:** Can we fit the simple PIB model to predict HOMO-LUMO gaps as accurately as SCF?

## 3.1 Defining the Problem

**For SCF (quantum mechanical model):**
- Number of electrons: $n_e = 6n_c + n_h$ (for alkenes: $n_e = 7n_c + 2$)
- HOMO index: $n_e/2 - 1$ (0-indexed)
- LUMO index: $n_e/2$
- Gap: $\Delta E^{HL} = E_{LUMO} - E_{HOMO}$

**For PIB model:**
- For $n_c$ carbons: HOMO has n = $n_c/2$, LUMO has n = $n_c/2 + 1$
- Box length: $L \approx 1.2 \times (n_c - 1)$ Angstroms

### Target Practice: Implement Gap Functions (15 points)

Complete the functions below to calculate HOMO-LUMO gaps for both models.

In [ ]:
def quantum_energy_gap(n_c):
    """
    Calculate HOMO-LUMO gap using SCF method.
    
    Args:
        n_c (int): Number of carbon atoms (must be even for alkenes)
        
    Returns:
        delta_e (float): HOMO-LUMO gap in Hartree
    """
    # YOUR CODE: Calculate electron count and orbital indices
    n_e = None         # Total electrons: 7*n_c + 2
    homo_index = None  # HOMO: n_e/2 - 1 (0-indexed)
    lumo_index = None  # LUMO: n_e/2
    
    # Build the molecule and run calculation
    iterations = n_c // 2
    smiles = "C=C" * iterations
    psi4_molecule = create_psi4_molecule(smiles)
    energy, wfn = psi4.energy('SCF/STO-3G', return_wfn=True, molecule=psi4_molecule)
    
    # Extract orbital energies and compute gap
    homo_e = wfn.epsilon_a().np[homo_index]
    lumo_e = wfn.epsilon_a().np[lumo_index]
    delta_e = lumo_e - homo_e
    return delta_e

In [ ]:
def pib_energy_gap(n_c):
    """
    Calculate HOMO-LUMO gap using particle-in-a-box model.
    
    Args:
        n_c (int): Number of carbon atoms
        
    Returns:
        delta_e (float): HOMO-LUMO gap in Hartree
    """
    # YOUR CODE: Calculate PIB quantum numbers and box length
    homo_n = None  # n_c / 2
    lumo_n = None  # n_c / 2 + 1
    L = None       # 1.2 * (n_c - 1)
    
    # Calculate energies using your pib_energy function
    pib_homo_e = pib_energy(homo_n, L)
    pib_lumo_e = None  # YOUR CODE
    delta_e = None     # YOUR CODE
    
    return delta_e

In [ ]:
# Calculate gaps for alkenes from 2 to 18 carbons
# This will take a minute or two to run the SCF calculations
print("Calculating HOMO-LUMO gaps for alkene series...")
quantum_gaps, pib_gaps = get_gaps(2, 18, quantum_energy_gap, pib_energy_gap)
print("Done!")

In [ ]:
chain_lengths = list(range(2, 19, 2))

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(chain_lengths, quantum_gaps, 'o-')
plt.title('HOMO-LUMO Gap: SCF Method')
plt.ylabel('Energy [Hartree]')
plt.xlabel('Number of Carbons')

plt.subplot(1, 2, 2)
plt.plot(chain_lengths, pib_gaps, 's-', color='orange')
plt.title('HOMO-LUMO Gap: PIB Model')
plt.ylabel('Energy [Hartree]')
plt.xlabel('Number of Carbons')

plt.tight_layout()
plt.show()

### Short Response Questions (10 points)

**Q3.1 (5 pts):** Compare the two plots. How similar or different are the predictions from each model?

**Q3.2 (5 pts):** Does the PIB model seem more valid for a certain range of chain lengths? Why might this be?

*Your answers here:*

Q3.1:

Q3.2:

---
## 3.2 Optimizing the Model

The raw PIB model doesn't match SCF well. Let's add **linear** and **constant** correction terms:

$$E^{HL}_{\text{fit}}(n_c) = \alpha \times E^{HL}_{\text{PIB}}(n_c) + \beta$$

We'll use `curve_fit` to find optimal values for $\alpha$ and $\beta$.

### Target Practice: Create the Corrected Model (10 points)

Write a function that applies the linear correction to the PIB energy gap.

In [ ]:
def corrected_pib_model(x, alpha, beta):
    """
    Apply linear correction to PIB energy gap.
    
    Args:
        x: Number of carbon atoms (array-like)
        alpha: Linear scaling factor
        beta: Constant offset
        
    Returns:
        Corrected energy gap
    """
    # YOUR CODE: Return alpha * pib_energy_gap(x) + beta
    # Note: x may be an array, so use np.vectorize if needed
    pib_gap = np.vectorize(pib_energy_gap)(x)
    y = None  # Your formula here
    return y

In [ ]:
# Fit the corrected model to SCF data
# We exclude n_c=2 (ethene) as it's an outlier for this model
x_data = np.array(range(4, 19, 2))  # 4 to 18 carbons
y_data = np.array(quantum_gaps[1:])  # Corresponding SCF gaps

params, covariance = curve_fit(
    corrected_pib_model,  # Function to fit
    x_data,               # x values (number of carbons)
    y_data,               # y values (SCF energy gaps)
    p0=[1, 0]             # Initial guesses for alpha, beta
)

print(f"Fitted parameters: alpha = {params[0]:.4f}, beta = {params[1]:.4f}")

In [ ]:
# Visualize the fit
pib_fit_plot(x_data, y_data, params, corrected_pib_model)

### Short Response Questions (10 points)

**Q3.3 (5 pts):** How well does the parameterized PIB model fit the SCF data, based on the $R^2$ value?

**Q3.4 (5 pts):** Would you expect this model to work well outside the range it was trained on (e.g., for very long chains)? Why or why not?

*Your answers here:*

Q3.3:

Q3.4:

---
# Reflection (20 points)

Answer these synthesis questions to demonstrate your understanding of the lab.

**Q4.1 (10 pts):** How do the boundary conditions of the particle-in-a-box system give rise to the discrete (quantized) energy levels? Be specific about the mathematical connection.

**Q4.2 (10 pts):** Explain at least two ways that simplified models (like PIB) and sophisticated models (like SCF) complement each other in computational chemistry. When would you use each?

*Your answers here:*

Q4.1:

Q4.2:

---
# References

1. Atkins, P., de Paula, J., & Keeler, J. (2018). *Atkins' Physical Chemistry* (11th ed.). Oxford University Press. Chapters 7-8.

2. Psi4 Development Team. (2023). Psi4: An open-source quantum chemistry program. https://psicode.org/

3. Landrum, G. (2023). RDKit: Open-source cheminformatics. https://www.rdkit.org/

4. SMILES Tutorial: https://www.daylight.com/dayhtml/doc/theory/theory.smiles.html